# **Importing libraries and dataset**

---



In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv('/content/Nat_Gas.csv')
df

,Dates,Prices
0,10/31/20,10.10
1,11/30/20,10.30
2,12/31/20,11.00
3,1/31/21,10.90
4,2/28/21,10.90
5,3/31/21,10.90
6,4/30/21,10.40
7,5/31/21,9.84
8,6/30/21,10.00
9,7/31/21,10.10


# **Parse dates and extracting month-year**

---



In [ ]:
df['Dates'] = pd.to_datetime(df['Dates'])
df['Month'] = df['Dates'].dt.month
df['Year'] = df['Dates'].dt.year
df['Month_name'] = df['Dates'].dt.strftime('%b')
df

/tmp/ipython-input-3404908582.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Dates'] = pd.to_datetime(df['Dates'])


,Dates,Prices,Month,Year,Month_name
0,2020-10-31,10.10,10,2020,Oct
1,2020-11-30,10.30,11,2020,Nov
2,2020-12-31,11.00,12,2020,Dec
3,2021-01-31,10.90,1,2021,Jan
4,2021-02-28,10.90,2,2021,Feb
5,2021-03-31,10.90,3,2021,Mar
6,2021-04-30,10.40,4,2021,Apr
7,2021-05-31,9.84,5,2021,May
8,2021-06-30,10.00,6,2021,Jun
9,2021-07-31,10.10,7,2021,Jul


# **Setting order for Month_Name column (getting rid of alphabetical ordering)**

---



In [ ]:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
df['Month_name'] = pd.Categorical(df['Month_name'], categories=month_order, ordered=True)
df

,Dates,Prices,Month,Year,Month_name
0,2020-10-31,10.10,10,2020,Oct
1,2020-11-30,10.30,11,2020,Nov
2,2020-12-31,11.00,12,2020,Dec
3,2021-01-31,10.90,1,2021,Jan
4,2021-02-28,10.90,2,2021,Feb
5,2021-03-31,10.90,3,2021,Mar
6,2021-04-30,10.40,4,2021,Apr
7,2021-05-31,9.84,5,2021,May
8,2021-06-30,10.00,6,2021,Jun
9,2021-07-31,10.10,7,2021,Jul


# **Compute seasonal average**

---



In [ ]:
monthly_avg = df.groupby('Month_name')['Prices'].mean().reset_index()
monthly_avg.columns = ['Month_name','Seasonal_Avg']
monthly_avg

/tmp/ipython-input-3807921971.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  monthly_avg = df.groupby('Month_name')['Prices'].mean().reset_index()


,Month_name,Seasonal_Avg
0,Jan,11.775
1,Feb,11.700
2,Mar,11.775
3,Apr,11.175
4,May,10.785
5,Jun,10.700
6,Jul,10.900
7,Aug,10.825
8,Sep,11.075
9,Oct,10.750


# **Merging monthly average into original table**

---



In [ ]:
df = df.merge(monthly_avg, on='Month_name')
df['Deviation'] = df['Prices'] - df['Seasonal_Avg']
df

,Dates,Prices,Month,Year,Month_name,Seasonal_Avg,Deviation
0,2020-10-31,10.10,10,2020,Oct,10.750,-0.650
1,2020-11-30,10.30,11,2020,Nov,11.325,-1.025
2,2020-12-31,11.00,12,2020,Dec,11.700,-0.700
3,2021-01-31,10.90,1,2021,Jan,11.775,-0.875
4,2021-02-28,10.90,2,2021,Feb,11.700,-0.800
5,2021-03-31,10.90,3,2021,Mar,11.775,-0.875
6,2021-04-30,10.40,4,2021,Apr,11.175,-0.775
7,2021-05-31,9.84,5,2021,May,10.785,-0.945
8,2021-06-30,10.00,6,2021,Jun,10.700,-0.700
9,2021-07-31,10.10,7,2021,Jul,10.900,-0.800


# **Create Flag based on deviation**

---



In [ ]:
def classify_signal(Deviation):
  if Deviation>0.2:
    return 'Bullish'
  elif Deviation<-0.2:
    return 'Bearish'
  else:
    return 'Neutral'

df['Signal'] = df['Deviation'].apply(classify_signal)
df

,Dates,Prices,Month,Year,Month_name,Seasonal_Avg,Deviation,Signal
0,2020-10-31,10.10,10,2020,Oct,10.750,-0.650,Bearish
1,2020-11-30,10.30,11,2020,Nov,11.325,-1.025,Bearish
2,2020-12-31,11.00,12,2020,Dec,11.700,-0.700,Bearish
3,2021-01-31,10.90,1,2021,Jan,11.775,-0.875,Bearish
4,2021-02-28,10.90,2,2021,Feb,11.700,-0.800,Bearish
5,2021-03-31,10.90,3,2021,Mar,11.775,-0.875,Bearish
6,2021-04-30,10.40,4,2021,Apr,11.175,-0.775,Bearish
7,2021-05-31,9.84,5,2021,May,10.785,-0.945,Bearish
8,2021-06-30,10.00,6,2021,Jun,10.700,-0.700,Bearish
9,2021-07-31,10.10,7,2021,Jul,10.900,-0.800,Bearish


# **Preprocess Data for Time Series**

---



In [ ]:
df = df.sort_values('Dates')
df.set_index('Dates', inplace=True)

# **Build Forecasting Model (SARIMAX / Holt-Winters / Prophet)**

---



In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Train the model
model = ExponentialSmoothing(df['Prices'], trend='add', seasonal='add', seasonal_periods=12)
fit_model = model.fit()

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency ME will be used.
  self._init_dates(dates, freq)


# **Forecast for 1 Year into the Future**

---



In [ ]:
# Forecast 12 months ahead
forecast = fit_model.forecast(12)

# **Create a Prediction Function**

---



In [ ]:
from datetime import datetime

def estimate_price(input_date_str):
    input_date = pd.to_datetime(input_date_str)
    full_series = pd.concat([fit_model.fittedvalues, forecast])

    if input_date in full_series.index:
        return round(full_series.loc[input_date], 2)
    else:
        # Interpolation for dates between existing records
        return round(np.interp(input_date.timestamp(),
                               full_series.index.astype(np.int64) / 1e9,
                               full_series.values), 2)


In [ ]:
estimate_price("2023-07-31")  # Past date

np.float64(11.17)

In [ ]:
estimate_price("2025-06-30")  # Future date

np.float64(12.05)